# Food production data – supplementary analysis

This notebook uses the crop file from the Food Bank dataset as a supplement to tasks 2 and 3. We only use rows where `Element` is `Production`. The values then have the same unit (tonnes), which makes comparisons meaningful.

The livestock file is not used because it contains several different units, and the Student Graduation dataset remains better suited to classification, scaling, train/test splitting, and PCA.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## 1. Load and inspect the data

The file is read in chunks to avoid loading the entire large dataset into memory at once.

In [2]:
data_path = Path("../data/crop1.csv")
if not data_path.exists():
    data_path = Path("data/crop1.csv")

parts = []
for chunk in pd.read_csv(data_path, chunksize=250_000):
    parts.append(chunk[chunk["Element"] == "Production"])

food = pd.concat(parts, ignore_index=True)
food = food.sort_values(["Area", "Item", "Year"]).reset_index(drop=True)
food.head()

,Area,Item,Element,Year,Unit,Value
0,Afghanistan,"Almonds, with shell",Production,1975,tonnes,0.0
1,Afghanistan,"Almonds, with shell",Production,1976,tonnes,9800.0
2,Afghanistan,"Almonds, with shell",Production,1977,tonnes,9000.0
3,Afghanistan,"Almonds, with shell",Production,1978,tonnes,12000.0
4,Afghanistan,"Almonds, with shell",Production,1979,tonnes,10500.0


In [3]:
print("Shape:", food.shape)
print("Years:", food["Year"].min(), "to", food["Year"].max())
print("Number of areas:", food["Area"].nunique())
print("Number of products:", food["Item"].nunique())
print("Units:", food["Unit"].unique())
print("Duplicate rows:", food.duplicated().sum())
food["Value"].describe()

Shape: (663735, 6)
Years: 1961 to 2020
Number of areas: 245
Number of products: 118
Units: <StringArray>
['tonnes']
Length: 1, dtype: str
Duplicate rows: 0


count    6.071180e+05
mean     2.478627e+06
std      2.333034e+07
min      0.000000e+00
25%      3.476000e+03
50%      3.322000e+04
75%      2.811800e+05
max      1.955308e+09
Name: Value, dtype: float64

## 2. Missing values

The data is annual time-series data for each area and product. A missing value between two known years is therefore estimated using linear interpolation within the same area and product. We do not extrapolate values before the first or after the last observation. Remaining missing rows are removed because there is not enough local information to estimate them reliably.

In [4]:
missing_before = food["Value"].isna().sum()

food["Value"] = food.groupby(["Area", "Item"], sort=False)["Value"].transform(
    lambda values: values.interpolate(method="linear", limit_area="inside")
)

missing_after = food["Value"].isna().sum()
food_clean = food.dropna(subset=["Value"]).copy()

print("Missing before interpolation:", missing_before)
print("Values filled by interpolation:", missing_before - missing_after)
print("Remaining rows removed:", missing_after)
print("Rows in cleaned data:", len(food_clean))

Missing before interpolation: 56617
Values filled by interpolation: 1494
Remaining rows removed: 55123
Rows in cleaned data: 608612


## 3a. Detect outliers

We use the IQR method because production is strongly skewed and the method is less affected by extreme values than a z-score. An IQR outlier is not automatically an error; a large country can genuinely produce much more than a small country.

In [5]:
q1 = food_clean["Value"].quantile(0.25)
q3 = food_clean["Value"].quantile(0.75)
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

food_clean["IQR_outlier"] = (
    (food_clean["Value"] < lower_limit)
    | (food_clean["Value"] > upper_limit)
)

print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)
print("IQR outliers:", food_clean["IQR_outlier"].sum())
print("Percentage:", round(food_clean["IQR_outlier"].mean() * 100, 2), "%")

Lower limit: -411500.0
Upper limit: 694900.0
IQR outliers: 102094
Percentage: 16.77 %


## 3b. Handle outliers

The high values are kept because they are plausible production values, not obvious measurement errors. Instead, `log1p` is used to reduce their influence while retaining all observations. It also works when production is zero. The original value is preserved for interpretation.

In [6]:
food_clean["Value_log"] = np.log1p(food_clean["Value"])

print("Skewness before log transformation:", round(food_clean["Value"].skew(), 2))
print("Skewness after log transformation:", round(food_clean["Value_log"].skew(), 2))
food_clean[["Value", "Value_log"]].describe()

Skewness before log transformation: 31.17
Skewness after log transformation: -0.23


,Value,Value_log
count,6.086120e+05,608612.000000
mean,2.472553e+06,10.283716
std,2.330201e+07,3.332313
min,0.000000e+00,0.000000
25%,3.400000e+03,8.131825
50%,3.300000e+04,10.404293
75%,2.800000e+05,12.542548
max,1.955308e+09,21.393813


## Conclusion

The Food Bank crop data gives meaningful work for tasks 2 and 3: missing values are partly interpolated within each time series, and valid extreme production values are handled with a log transformation. No categorical encoding, scaling, train/test split, or PCA is performed here because the Student Graduation dataset remains the main dataset for the modelling-related tasks.